In [31]:




import pandas as pd
from pathlib import Path

import os

PROJECT_ROOT = Path(
    os.getenv("PROJECT_ROOT", Path.cwd())
).expanduser().resolve()


JUDGE_PATH8 = (
    PROJECT_ROOT


    / "llm_judge_results_answers_log_llmaindex_nonempty_rows_1_to_50.csv"
)


CSV_PATH_NEW8 =  Path(os.getenv("ANSWERS_LOG_PATH", str(JUDGE_PATH8))).expanduser().resolve()
df_new_8 = pd.read_csv(CSV_PATH_NEW8)






In [32]:



import pandas as pd
from pathlib import Path

import os

PROJECT_ROOT = Path(
    os.getenv("PROJECT_ROOT", Path.cwd())
).expanduser().resolve()


JUDGE_PATH7 = (
    PROJECT_ROOT


    / "llm_judge_results_answers_log_new_dataset_8_part4_rows_1_to_370.csv"
)


CSV_PATH_NEW7 =  Path(os.getenv("ANSWERS_LOG_PATH", str(JUDGE_PATH7))).expanduser().resolve()
df_new_7 = pd.read_csv(CSV_PATH_NEW7)






In [33]:




import pandas as pd
from pathlib import Path

import os

PROJECT_ROOT = Path(
    os.getenv("PROJECT_ROOT", Path.cwd())
).expanduser().resolve()


JUDGE_PATH6 = (
    PROJECT_ROOT


    / "llm_judge_results_answers_log_new_dataset_8_part3_rows_1_to_299.csv"
)


CSV_PATH_NEW6 =  Path(os.getenv("ANSWERS_LOG_PATH", str(JUDGE_PATH6))).expanduser().resolve()
df_new_6 = pd.read_csv(CSV_PATH_NEW6)






In [34]:



import pandas as pd
from pathlib import Path

import os

PROJECT_ROOT = Path(
    os.getenv("PROJECT_ROOT", Path.cwd())
).expanduser().resolve()


JUDGE_PATH5 = (
    PROJECT_ROOT


    / "llm_judge_results_answers_log_new_dataset_8_part2_rows_1_to_400.csv"
)


CSV_PATH_NEW5 =  Path(os.getenv("ANSWERS_LOG_PATH", str(JUDGE_PATH5))).expanduser().resolve()
df_new_5 = pd.read_csv(CSV_PATH_NEW5)






In [35]:

import pandas as pd
from pathlib import Path

import os

PROJECT_ROOT = Path(
    os.getenv("PROJECT_ROOT", Path.cwd())
).expanduser().resolve()


JUDGE_PATH4 = (
    PROJECT_ROOT


    / "llm_judge_results_answers_log_new_dataset_8_rows_1_to_500.csv"
)


CSV_PATH_NEW4 =  Path(os.getenv("ANSWERS_LOG_PATH", str(JUDGE_PATH4))).expanduser().resolve()
df_new_4 = pd.read_csv(CSV_PATH_NEW4)






In [36]:
import pandas as pd
from pathlib import Path

import os

PROJECT_ROOT = Path(
    os.getenv("PROJECT_ROOT", Path.cwd())
).expanduser().resolve()


JUDGE_PATH3 = (
    PROJECT_ROOT


    / "llm_judge_results_answers_log_new_dataset_8_rows_1_to_402.csv"
)


CSV_PATH_NEW3 =  Path(os.getenv("ANSWERS_LOG_PATH", str(JUDGE_PATH3))).expanduser().resolve()
df_new_3 = pd.read_csv(CSV_PATH_NEW3)





In [37]:
import pandas as pd
from pathlib import Path

import os

PROJECT_ROOT = Path(
    os.getenv("PROJECT_ROOT", Path.cwd())
).expanduser().resolve()


JUDGE_PATH2 = (
    PROJECT_ROOT


    / "llm_judge_results_answers_log_new_dataset_rows_1_to_289.csv"
)


CSV_PATH_NEW2 =  Path(os.getenv("ANSWERS_LOG_PATH", str(JUDGE_PATH2))).expanduser().resolve()
df_new_2 = pd.read_csv(CSV_PATH_NEW2)





In [38]:
import pandas as pd
from pathlib import Path

import os

PROJECT_ROOT = Path(
    os.getenv("PROJECT_ROOT", Path.cwd())
).expanduser().resolve()


JUDGE_PATH = (
    PROJECT_ROOT


    / "llm_judge_results_answers_log_new_dataset_8_rows_1_to_400.csv"
)


CSV_PATH_NEW =  Path(os.getenv("ANSWERS_LOG_PATH", str(JUDGE_PATH))).expanduser().resolve()
df_new_1 = pd.read_csv(CSV_PATH_NEW)





In [39]:
df_new = (
    pd.concat(
        [df_new_8,df_new_7, df_new_6, df_new_5, df_new_4, df_new_3, df_new_2, df_new_1],
        axis=0
    )
    .drop_duplicates()
)


In [40]:
rows_per_script = df_new.groupby("script").size()
print(rows_per_script)


script
LLMGraph_Hybrid_Community_Retriever           190
LLMGraph_Hybrid_Retriever                     380
LLMGraph_Vector_KG_Retriever                  190
LlamaIndex_BM25_Hybrid_Retriever_Rerank       186
LlamaIndex_BM25_Vector_PG_Community_Rerank    143
LlamaIndex_Vector_KG_Retriever                190
RAG_Advanced_Dense                            190
RAG_Advanced_Hybrid                           190
RAG_Advanced_Sparse                           192
RAG_Naive_Hybrid                              190
SimpleKG_Hybrid_Community_Retriever_Rerank    190
SimpleKG_Hybrid_KG_Retriever                  100
SimpleKG_Hybrid_Retriever                      90
SimpleKG_Hybrid_Retriever_Rerank              100
SimpleKG_Vector_KG_Retriever                  189
dtype: int64


In [42]:
df= df_new[df_new["script"] == "LLMGraph_Hybrid_Retriever"]
import pandas as pd



# 1) Priorität für correctness_category festlegen (höher = besser)
# TP soll gewinnen, danach FP, danach alles andere
category_rank = {"TP": 2, "FP": 1}

df["correctness_rank"] = df["correctness_category"].map(category_rank).fillna(0).astype(int)

# 2) Scores robust in Zahlen umwandeln (falls sie als Strings vorliegen)
df["faithfulness_1to5"] = pd.to_numeric(df["faithfulness_1to5"], errors="coerce").fillna(-1)
df["answer_relevance_1to5"] = pd.to_numeric(df["answer_relevance_1to5"], errors="coerce").fillna(-1)

# (Optional) weiterer Tie-Breaker, falls du willst:
df["helpfulness_final_1to5"] = pd.to_numeric(df["helpfulness_final_1to5"], errors="coerce").fillna(-1)

# 3) Sortieren: pro (script, question_id) die beste Zeile nach Ranking zuerst
df_sorted = df.sort_values(
    by=[
        "script",
        "question_id",
        "correctness_rank",      # TP > FP > Rest
        "faithfulness_1to5",     # höher besser
        "answer_relevance_1to5", # höher besser
        "helpfulness_final_1to5" # optional tie-breaker
    ],
    ascending=[True, True, False, False, False, False],
)

# 4) Pro (script, question_id) nur die erste (beste) Zeile behalten
df_best = df_sorted.drop_duplicates(subset=["script", "question_id"], keep="first")

df_best


C:\Users\Nasiba\AppData\Local\Temp\ipykernel_34632\2495633306.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["correctness_rank"] = df["correctness_category"].map(category_rank).fillna(0).astype(int)
C:\Users\Nasiba\AppData\Local\Temp\ipykernel_34632\2495633306.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["faithfulness_1to5"] = pd.to_numeric(df["faithfulness_1to5"], errors="coerce").fillna(-1)
C:\Users\Nasiba\AppData\Local\Temp\ipykernel_34632\2495633306.py:14: SettingWithCopyWarning: 
A 

,script,question_id,query_type,answer_relevance_score,answer_relevance_1to5,completeness_score,completeness_1to5,correctness_category,correctness_coverage,correctness_coverage_1to5,correctness_error_severity,faithfulness_score,faithfulness_1to5,helpfulness_raw_1to5,helpfulness_raw_score,helpfulness_final_score,helpfulness_final_1to5,helpfulness_justification,metrics_json,correctness_rank
119,LLMGraph_Hybrid_Retriever,1,factual,0.801728,4,1.000000,5,TP,1.000000,5,low,1.000000,5,5,1.0,1.000000,5,The answer directly addresses the question by ...,"{""answer_relevance"": {""score"": 0.8017280697822...",2
120,LLMGraph_Hybrid_Retriever,2,factual,0.842208,4,1.000000,5,TP,1.000000,5,low,1.000000,5,5,1.0,1.000000,5,The answer directly addresses the question by ...,"{""answer_relevance"": {""score"": 0.8422080874443...",2
121,LLMGraph_Hybrid_Retriever,3,relational,0.777847,4,0.250000,2,FP,0.333333,2,high,0.250000,2,5,1.0,0.625000,4,The answer directly addresses the question by ...,"{""answer_relevance"": {""score"": 0.7778468926747...",1
122,LLMGraph_Hybrid_Retriever,4,relational,0.524356,3,1.000000,5,TP,1.000000,5,low,1.000000,5,5,1.0,1.000000,5,The answer directly addresses the question by ...,"{""answer_relevance"": {""score"": 0.5243559479713...",2
123,LLMGraph_Hybrid_Retriever,5,summary,0.858646,4,0.600000,3,TP,1.000000,5,low,1.000000,5,5,1.0,1.000000,5,The answer provides a clear and detailed summa...,"{""answer_relevance"": {""score"": 0.8586464921633...",2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,LLMGraph_Hybrid_Retriever,186,summary,0.653757,4,0.333333,2,TP,1.000000,5,low,0.350000,2,5,1.0,0.675000,4,The answer directly addresses the question by ...,"{""answer_relevance"": {""score"": 0.6537567973136...",2
6,LLMGraph_Hybrid_Retriever,187,summary,0.576888,3,0.600000,3,PARTIAL,0.400000,3,low,0.416667,3,5,1.0,0.708333,4,The answer directly addresses the question by ...,"{""answer_relevance"": {""score"": 0.5768879850705...",0
7,LLMGraph_Hybrid_Retriever,188,summary,0.669926,4,0.750000,4,TP,1.000000,5,low,0.416667,3,5,1.0,0.708333,4,The answer directly addresses the question by ...,"{""answer_relevance"": {""score"": 0.6699263453483...",2
8,LLMGraph_Hybrid_Retriever,189,summary,0.737156,4,0.666667,4,PARTIAL,0.670000,4,low,0.916667,5,5,1.0,0.958333,5,The answer directly addresses the question by ...,"{""answer_relevance"": {""score"": 0.7371564904848...",0


In [43]:
check = df_best.groupby(["script", "question_id"]).size().value_counts()
print(check)


1    190
Name: count, dtype: int64


In [44]:
df_newest = df_new[df_new["script"] != "LLMGraph_Hybrid_Retriever"]
df_newest_combi = (
    pd.concat(
        [df_newest,df_best ],
        axis=0
    )
    .drop_duplicates()
)

In [ ]:
check = df_best.groupby(["script", "question_id"]).size().value_counts()
print(check)


In [45]:
counts = (
    df_newest_combi.groupby(["script", "correctness_error_severity"])
      .size()
      .unstack(fill_value=0)
)
print(counts)


correctness_error_severity                  high  low  medium
script                                                       
LLMGraph_Hybrid_Community_Retriever           10  173       7
LLMGraph_Hybrid_Retriever                     16  169       5
LLMGraph_Vector_KG_Retriever                   8  170      12
LlamaIndex_BM25_Hybrid_Retriever_Rerank       11  162      13
LlamaIndex_BM25_Vector_PG_Community_Rerank     9  128       6
LlamaIndex_Vector_KG_Retriever                12  166      12
RAG_Advanced_Dense                            22  159       9
RAG_Advanced_Hybrid                           15  158      17
RAG_Advanced_Sparse                           13  161      18
RAG_Naive_Hybrid                              16  171       3
SimpleKG_Hybrid_Community_Retriever_Rerank     9  176       5
SimpleKG_Hybrid_KG_Retriever                   7   88       5
SimpleKG_Hybrid_Retriever                      6   82       2
SimpleKG_Hybrid_Retriever_Rerank               6   90       4
SimpleKG

In [46]:
counts = (
    df_newest_combi.groupby(["script", "correctness_category"])
      .size()
      .unstack(fill_value=0)
)

counts = counts.reindex(columns=["TP", "FP", "FN"], fill_value=0)

# nach Anzahl TP 
counts = counts.sort_values(by="TP", ascending=False)

print(counts)


correctness_category                         TP  FP   FN
script                                                  
SimpleKG_Hybrid_Community_Retriever_Rerank  141   9   23
LLMGraph_Hybrid_Retriever                   140  14   15
LLMGraph_Hybrid_Community_Retriever         129  10   18
RAG_Advanced_Sparse                         129  12    8
RAG_Advanced_Hybrid                         124  15   11
SimpleKG_Vector_KG_Retriever                121  16   29
LlamaIndex_BM25_Hybrid_Retriever_Rerank     121  11   21
LLMGraph_Vector_KG_Retriever                117   8   29
LlamaIndex_BM25_Vector_PG_Community_Rerank   91   8   27
LlamaIndex_Vector_KG_Retriever               83  10   62
SimpleKG_Hybrid_Retriever_Rerank             70   5   13
SimpleKG_Hybrid_KG_Retriever                 65   7   17
SimpleKG_Hybrid_Retriever                    64   5    6
RAG_Naive_Hybrid                             17  16  130
RAG_Advanced_Dense                           12  21  137


In [47]:
tp_counts = (
    df_newest_combi[df_newest_combi["correctness_category"] == "TP"]
        .groupby(["script", "query_type"])
        .size()
        .unstack(fill_value=0)
)

# optional: nach summary-TP sortieren
tp_counts = tp_counts.sort_values(by="factual", ascending=False)

print(tp_counts.to_string())


query_type                                  disambiguation  factual  multi_hop  reasoning  relational  summary
script                                                                                                        
RAG_Advanced_Sparse                                     23       28         18         19          25       16
RAG_Advanced_Hybrid                                     22       27         16         20          25       14
LLMGraph_Hybrid_Community_Retriever                     19       20         21         21          21       27
LLMGraph_Hybrid_Retriever                               21       20         24         24          22       29
SimpleKG_Hybrid_Community_Retriever_Rerank              21       19         25         25          22       29
LlamaIndex_BM25_Hybrid_Retriever_Rerank                 19       19         23         20          19       21
LLMGraph_Vector_KG_Retriever                            20       19         18         22          18       20
S

In [48]:
tp_counts = (
    df_newest_combi[df_newest_combi["correctness_category"] == "TP"]
        .groupby(["script", "query_type"])
        .size()
        .unstack(fill_value=0)
)

# optional: nach summary-TP sortieren
tp_counts = tp_counts.sort_values(by="multi_hop", ascending=False)

print(tp_counts.to_string())


query_type                                  disambiguation  factual  multi_hop  reasoning  relational  summary
script                                                                                                        
SimpleKG_Hybrid_Community_Retriever_Rerank              21       19         25         25          22       29
SimpleKG_Hybrid_Retriever                               16        1         24          9           4       10
LLMGraph_Hybrid_Retriever                               21       20         24         24          22       29
SimpleKG_Vector_KG_Retriever                            20       19         24         20          20       18
LlamaIndex_BM25_Hybrid_Retriever_Rerank                 19       19         23         20          19       21
LLMGraph_Hybrid_Community_Retriever                     19       20         21         21          21       27
LLMGraph_Vector_KG_Retriever                            20       19         18         22          18       20
R

In [50]:
tp_counts = (
    df_newest_combi[df_newest_combi["correctness_category"] == "TP"]
        .groupby(["script", "query_type"])
        .size()
        .unstack(fill_value=0)
)

# optional: nach summary-TP sortieren
tp_counts = tp_counts.sort_values(by="relational", ascending=False)

print(tp_counts.to_string())


query_type                                  disambiguation  factual  multi_hop  reasoning  relational  summary
script                                                                                                        
RAG_Advanced_Hybrid                                     22       27         16         20          25       14
RAG_Advanced_Sparse                                     23       28         18         19          25       16
LLMGraph_Hybrid_Retriever                               21       20         24         24          22       29
SimpleKG_Hybrid_Community_Retriever_Rerank              21       19         25         25          22       29
LLMGraph_Hybrid_Community_Retriever                     19       20         21         21          21       27
SimpleKG_Vector_KG_Retriever                            20       19         24         20          20       18
LlamaIndex_BM25_Hybrid_Retriever_Rerank                 19       19         23         20          19       21
S

In [51]:
tp_counts = (
    df_newest_combi[df_newest_combi["correctness_category"] == "TP"]
        .groupby(["script", "query_type"])
        .size()
        .unstack(fill_value=0)
)

# optional: nach summary-TP sortieren
tp_counts = tp_counts.sort_values(by="summary", ascending=False)

print(tp_counts.to_string())


query_type                                  disambiguation  factual  multi_hop  reasoning  relational  summary
script                                                                                                        
LLMGraph_Hybrid_Retriever                               21       20         24         24          22       29
SimpleKG_Hybrid_Community_Retriever_Rerank              21       19         25         25          22       29
LLMGraph_Hybrid_Community_Retriever                     19       20         21         21          21       27
LlamaIndex_BM25_Hybrid_Retriever_Rerank                 19       19         23         20          19       21
LLMGraph_Vector_KG_Retriever                            20       19         18         22          18       20
SimpleKG_Vector_KG_Retriever                            20       19         24         20          20       18
LlamaIndex_BM25_Vector_PG_Community_Rerank              14       18         13         15          15       16
R

In [52]:


cols = ["script",  "faithfulness_1to5", "answer_relevance_1to5","completeness_1to5","correctness_coverage_1to5" ,"helpfulness_final_1to5" ]
df_new_scores = df_newest_combi[cols].copy()



summary = (
    df_new_scores.groupby("script", as_index=False)
      .agg(
          Faith=("faithfulness_1to5", "mean"),
          Ans_Rel=("answer_relevance_1to5", "mean"),
          Cont_Rel=("completeness_1to5", "mean"),
          Corr =("correctness_coverage_1to5", "mean"),  
              Help =("helpfulness_final_1to5", "mean"),  
          
        
          N=("script", "size"),
      )
)


summary_no_n = summary.drop(columns=["N"])


summary_no_n = summary_no_n.set_index("script").round(2)

print(summary_no_n.to_string())


                                            Faith  Ans_Rel  Cont_Rel  Corr  Help
script                                                                          
LLMGraph_Hybrid_Community_Retriever          3.85     3.71      3.03  4.03  4.20
LLMGraph_Hybrid_Retriever                    4.01     3.77      3.04  4.12  4.28
LLMGraph_Vector_KG_Retriever                 4.60     3.61      2.78  3.82  4.43
LlamaIndex_BM25_Hybrid_Retriever_Rerank      4.67     3.66      3.04  3.94  4.51
LlamaIndex_BM25_Vector_PG_Community_Rerank   4.69     3.61      2.95  3.78  4.35
LlamaIndex_Vector_KG_Retriever               4.72     3.56      2.35  3.07  3.98
RAG_Advanced_Dense                           4.24     3.39      1.41  1.43  1.92
RAG_Advanced_Hybrid                          4.78     3.77      2.85  3.99  4.58
RAG_Advanced_Sparse                          4.77     3.80      2.97  4.06  4.64
RAG_Naive_Hybrid                             4.37     2.27      1.35  1.63  2.15
SimpleKG_Hybrid_Community_Re

In [53]:
# -------------------------------
# 1) Spalten definieren
# -------------------------------
cols = [
    "script",
    "helpfulness_raw_1to5",
    "helpfulness_raw_score",
    "helpfulness_final_score",
    "helpfulness_final_1to5",
    "helpfulness_justification",
]

# -------------------------------
# 2) Relevante Daten auswählen
# -------------------------------
df_new_scores = df_newest_combi[cols].copy()

# -------------------------------
# 3) Mittelwerte pro Script berechnen
# -------------------------------
summary = (
    df_new_scores
        .groupby("script")
        .mean(numeric_only=True)
        .round(2)
)

# -------------------------------
# 4) (Optional) Anzahl Einträge pro Script
# -------------------------------
summary["N"] = df_new_scores.groupby("script").size()

# -------------------------------
# 5) Ausgabe
# -------------------------------
print(summary.to_string())


                                            helpfulness_raw_1to5  helpfulness_raw_score  helpfulness_final_score  helpfulness_final_1to5    N
script                                                                                                                                       
LLMGraph_Hybrid_Community_Retriever                         4.73                   0.93                     0.80                    4.20  190
LLMGraph_Hybrid_Retriever                                   4.73                   0.93                     0.81                    4.28  190
LLMGraph_Vector_KG_Retriever                                4.51                   0.88                     0.84                    4.43  190
LlamaIndex_BM25_Hybrid_Retriever_Rerank                     4.59                   0.90                     0.86                    4.51  186
LlamaIndex_BM25_Vector_PG_Community_Rerank                  4.41                   0.85                     0.82                    4.35  143
LlamaI

In [54]:
summary_by_query_type = (
    df_newest_combi
        .groupby("query_type", as_index=False)
        .agg(
            Faith=("faithfulness_1to5", "mean"),
            Ans_Rel=("answer_relevance_1to5", "mean"),
            Cont_Rel=("completeness_1to5", "mean"),
            Corr=("correctness_coverage_1to5", "mean"),
            Help=("helpfulness_final_1to5", "mean"),
            N=("query_type", "size"),
        )
        .set_index("query_type")
        .round(2)
)

print(summary_by_query_type.to_string())


                Faith  Ans_Rel  Cont_Rel  Corr  Help    N
query_type                                               
disambiguation   4.40     3.61      2.69  3.52  4.16  390
factual          4.52     3.73      2.96  3.42  3.75  421
multi_hop        4.19     3.55      2.75  3.88  4.16  378
reasoning        4.10     3.50      2.51  3.60  4.04  397
relational       4.36     3.35      2.77  3.53  3.66  410
summary          4.30     3.62      2.45  3.29  4.02  524


In [55]:
summary_by_script_and_query = (
    df_newest_combi 
        .groupby(["script", "query_type"], as_index=False)
        .agg(
            Faith=("faithfulness_1to5", "mean"),
            Ans_Rel=("answer_relevance_1to5", "mean"),
            Cont_Rel=("completeness_1to5", "mean"),
            Corr=("correctness_coverage_1to5", "mean"),
            Help=("helpfulness_final_1to5", "mean"),
            N=("script", "size"),
        )
        .set_index(["script", "query_type"])
        .round(2)
)

print(summary_by_script_and_query.to_string())


                                                           Faith  Ans_Rel  Cont_Rel  Corr  Help   N
script                                     query_type                                              
LLMGraph_Hybrid_Community_Retriever        disambiguation   3.87     3.60      3.43  3.87  4.23  30
                                           factual          4.30     3.87      3.57  3.87  4.20  30
                                           multi_hop        3.83     3.53      2.77  4.20  4.33  30
                                           reasoning        3.60     3.73      2.90  4.10  4.20  30
                                           relational       3.90     3.73      3.07  4.10  4.10  30
                                           summary          3.68     3.75      2.60  4.05  4.15  40
LLMGraph_Hybrid_Retriever                  disambiguation   4.03     3.73      2.80  3.93  4.27  30
                                           factual          4.60     4.13      3.50  3.80  4.33  30


In [56]:
pivot_help = (
    summary_by_script_and_query
        .reset_index()
        .pivot(
            index="script",
            columns="query_type",
            values="Help"
        )
        .round(2)
)

# nach der Spalte "summary" sortieren (absteigend)
pivot_help = pivot_help.sort_values(by="summary", ascending=False)

print(pivot_help.to_string())


query_type                                  disambiguation  factual  multi_hop  reasoning  relational  summary
script                                                                                                        
RAG_Advanced_Sparse                                   4.70     4.88       4.53       4.33        4.63     4.70
LlamaIndex_BM25_Hybrid_Retriever_Rerank               4.77     4.07       4.90       4.57        4.03     4.67
LLMGraph_Vector_KG_Retriever                          4.53     3.87       4.73       4.77        3.97     4.62
LlamaIndex_BM25_Vector_PG_Community_Rerank            4.71     4.17       4.87       4.35        3.81     4.52
RAG_Advanced_Hybrid                                   4.80     4.67       4.40       4.53        4.67     4.45
LLMGraph_Hybrid_Retriever                             4.27     4.33       4.27       4.27        4.10     4.40
SimpleKG_Hybrid_Retriever                             4.52     5.00       4.30       4.07        4.17     4.33
S

In [57]:
pivot_help = (
    summary_by_script_and_query
        .reset_index()
        .pivot(
            index="script",
            columns="query_type",
            values="Faith"
        )
        .round(2)
)


pivot_help = pivot_help.sort_values(by="summary", ascending=False)

print(pivot_help.to_string())


query_type                                  disambiguation  factual  multi_hop  reasoning  relational  summary
script                                                                                                        
RAG_Advanced_Sparse                                   4.90     4.88       4.70       4.53        4.67     4.90
LlamaIndex_Vector_KG_Retriever                        4.50     4.67       4.83       4.60        4.83     4.82
RAG_Advanced_Hybrid                                   4.93     4.90       4.50       4.77        4.83     4.75
LlamaIndex_BM25_Hybrid_Retriever_Rerank               4.87     4.77       4.80       4.32        4.48     4.72
LLMGraph_Vector_KG_Retriever                          4.70     4.50       4.77       4.57        4.37     4.68
LlamaIndex_BM25_Vector_PG_Community_Rerank            4.90     4.70       4.87       4.35        4.78     4.59
RAG_Advanced_Dense                                    4.17     4.30       4.00       4.03        4.23     4.58
R

In [58]:
pivot_help = (
    summary_by_script_and_query
        .reset_index()
        .pivot(
            index="script",
            columns="query_type",
            values="Ans_Rel"
        )
        .round(2)
)


pivot_help = pivot_help.sort_values(by="summary", ascending=False)

print(pivot_help.to_string())


query_type                                  disambiguation  factual  multi_hop  reasoning  relational  summary
script                                                                                                        
SimpleKG_Hybrid_Retriever_Rerank                      3.89     4.17       3.33       3.56        3.58     3.95
SimpleKG_Hybrid_KG_Retriever                          3.89     4.07       3.33       3.50        3.25     3.95
LlamaIndex_BM25_Vector_PG_Community_Rerank            3.67     3.80       3.47       3.30        3.37     3.93
LLMGraph_Hybrid_Retriever                             3.73     4.13       3.67       3.57        3.67     3.85
LLMGraph_Vector_KG_Retriever                          3.63     3.87       3.67       3.57        3.07     3.80
SimpleKG_Hybrid_Community_Retriever_Rerank            3.90     4.03       3.57       3.57        3.43     3.78
RAG_Advanced_Sparse                                   3.73     4.06       3.53       3.80        3.90     3.78
L

In [59]:
pivot_help = (
    summary_by_script_and_query
        .reset_index()
        .pivot(
            index="script",
            columns="query_type",
            values="Cont_Rel"
        )
        .round(2)
)


pivot_help = pivot_help.sort_values(by="summary", ascending=False)

print(pivot_help.to_string())


query_type                                  disambiguation  factual  multi_hop  reasoning  relational  summary
script                                                                                                        
SimpleKG_Hybrid_Retriever_Rerank                      2.56     3.55       3.33       3.12        3.17     3.00
LLMGraph_Hybrid_Retriever                             2.80     3.50       2.77       3.00        3.27     2.92
LlamaIndex_BM25_Vector_PG_Community_Rerank            3.19     3.00       3.40       2.61        2.96     2.74
LlamaIndex_BM25_Hybrid_Retriever_Rerank               2.77     3.47       3.30       2.93        3.14     2.74
RAG_Advanced_Sparse                                   3.07     3.59       2.97       2.50        3.07     2.70
LLMGraph_Vector_KG_Retriever                          2.53     3.13       2.90       2.60        3.00     2.60
LLMGraph_Hybrid_Community_Retriever                   3.43     3.57       2.77       2.90        3.07     2.60
S

In [60]:
pivot_help = (
    summary_by_script_and_query
        .reset_index()
        .pivot(
            index="script",
            columns="query_type",
            values="Corr"
        )
        .round(2)
)


pivot_help = pivot_help.sort_values(by="summary", ascending=False)

print(pivot_help.to_string())


query_type                                  disambiguation  factual  multi_hop  reasoning  relational  summary
script                                                                                                        
LLMGraph_Hybrid_Retriever                             3.93     3.80       4.37       4.23        4.10     4.22
SimpleKG_Hybrid_Community_Retriever_Rerank            3.97     3.67       4.37       4.30        4.03     4.15
SimpleKG_Hybrid_Retriever_Rerank                      3.67     3.76       5.00       4.31        4.04     4.11
LLMGraph_Hybrid_Community_Retriever                   3.87     3.87       4.20       4.10        4.10     4.05
LlamaIndex_BM25_Hybrid_Retriever_Rerank               3.80     3.67       4.43       4.07        3.83     3.85
LlamaIndex_BM25_Vector_PG_Community_Rerank            4.05     3.60       4.73       3.65        3.52     3.59
LLMGraph_Vector_KG_Retriever                          3.87     3.73       4.03       4.13        3.63     3.58
S